# rearcvac: escala no vista sobre reglas conocidas (ver docs/PREREG_nemvac.md, segunda parte)


In [ ]:
%%writefile /kaggle/working/_worker_src.py
import json, os, sys, time
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
T00 = time.time()
SHARD = int(os.environ["SHARD"]); N_SHARDS = 4
import numpy as np
import torch
from torch.nn.attention import SDPBackend, sdpa_kernel
from transformers import AutoModelForCausalLM

DATA = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "arc-agi_training_challenges.json" in files:
        DATA = root
        break
assert DATA, "sin dataset de la competicion"
TR = json.load(open(os.path.join(DATA, "arc-agi_training_challenges.json")))
TRS = json.load(open(os.path.join(DATA, "arc-agi_training_solutions.json")))
EV = json.load(open(os.path.join(DATA, "arc-agi_evaluation_challenges.json")))
EVS = json.load(open(os.path.join(DATA, "arc-agi_evaluation_solutions.json")))
NEMF = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "nem.json" in files:
        NEMF = os.path.join(root, "nem.json")
        break
assert NEMF, "sin nem.json (dataset arc-agi2-nemvac-data)"
P = json.load(open(NEMF))

ITEMS = list(P["nem"])
for grupo, ks in P["ctrl"].items():
    for k in ks:
        ch, so = (EV, EVS) if grupo == "E_eval" else (TR, TRS)
        ITEMS.append({"id": k, "src": grupo, "train": ch[k]["train"],
                      "test_input": ch[k]["test"][0]["input"], "gt": so[k][0]})
MIOS = ITEMS[SHARD::N_SHARDS]

MDIR = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "config.json" in files and any(x.endswith(".safetensors") for x in files):
        MDIR = root
        break
assert MDIR
_SDPA = [SDPBackend.EFFICIENT_ATTENTION, SDPBackend.FLASH_ATTENTION, SDPBackend.MATH]
IM_START, USER, ASSIST, NL, EOS = 14, 11, 12, 10, 15
MAX_TOK = 8000


def grid_tokens(g):
    out = []
    for i, row in enumerate(g):
        out.extend(int(c) for c in row)
        if i < len(g) - 1:
            out.append(NL)
    return out


def par(inp, out):
    return ([IM_START, USER, NL] + grid_tokens(inp) + [EOS] + [IM_START, ASSIST, NL]
            + grid_tokens(out) + [EOS])


def query(inp):
    return [IM_START, USER, NL] + grid_tokens(inp) + [EOS] + [IM_START, ASSIST, NL]


MODEL = AutoModelForCausalLM.from_pretrained(MDIR, torch_dtype=torch.bfloat16, device_map="cuda:0",
                                             attn_implementation="sdpa")
MODEL.eval()
print("[s%d] modelo listo (%.0fs), %d items" % (SHARD, time.time() - T00, len(MIOS)), flush=True)


@torch.no_grad()
def nll(it):
    demos = [par(p["input"], p["output"]) for p in it["train"]]
    q = query(it["test_input"])
    gt = grid_tokens(it["gt"]) + [EOS]
    while len(demos) > 1 and sum(map(len, demos)) + len(q) + len(gt) > MAX_TOK:
        demos.pop(0)
    prompt = [t for d in demos for t in d] + q
    if len(prompt) + len(gt) > MAX_TOK:
        return None
    ids = torch.tensor([prompt + gt], device="cuda:0", dtype=torch.long)
    with sdpa_kernel(_SDPA):
        h = MODEL.model(input_ids=ids).last_hidden_state[0, len(prompt) - 1:len(prompt) - 1 + len(gt)]
    lp = MODEL.lm_head(h).float().log_softmax(-1)
    tgt = ids[0, len(prompt):len(prompt) + len(gt)]
    v = -lp.gather(1, tgt.view(-1, 1)).squeeze(1)
    celdas = len(it["gt"]) * len(it["gt"][0])
    return {"nll": round(float(v.sum()), 4), "celdas": celdas,
            "nll_celda": round(float(v.sum()) / celdas, 6),
            "greedy_ok": bool((lp.argmax(-1) == tgt).all().item()),
            "n_demos": len(demos), "n_demos_total": len(it["train"]), "n_tok": len(prompt) + len(gt)}


res = []
for i, it in enumerate(MIOS):
    try:
        r = nll(it)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache(); r = {"oom": True}
    if r is not None:
        r.update({"id": it["id"], "src": it["src"]})
        res.append(r)
    if i % 20 == 0:
        print("[s%d] %d/%d (%.0fs)" % (SHARD, i, len(MIOS), time.time() - T00), flush=True)
json.dump(res, open("/kaggle/working/shard%d.json" % SHARD, "w"))
print("[s%d] fin %d resultados (%.0fs)" % (SHARD, len(res), time.time() - T00), flush=True)


In [ ]:
import json, os, subprocess, sys, time, math
T0 = time.time()
procs = []
for i in range(4):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(i), SHARD=str(i))
    fh = open("/kaggle/working/log%d.txt" % i, "w")
    procs.append((i, subprocess.Popen([sys.executable, "/kaggle/working/_worker_src.py"], env=env,
                                      stdout=fh, stderr=subprocess.STDOUT), fh))
for i, p, fh in procs:
    p.wait(); fh.close()
    print("worker %d rc=%d (%.0fs)" % (i, p.returncode, time.time() - T0), flush=True)
res = []
for i in range(4):
    f = "/kaggle/working/shard%d.json" % i
    if os.path.exists(f):
        res += json.load(open(f))
    print("== log %d ==" % i); print(open("/kaggle/working/log%d.txt" % i).read()[-1500:], flush=True)
json.dump(res, open("/kaggle/working/rearcvac.json", "w"))

def med(x):
    s = sorted(x); n = len(s)
    return float("nan") if not n else (s[n // 2] if n % 2 else (s[n // 2 - 1] + s[n // 2]) / 2)

print("")
print("=" * 96)
print("NEMVAC: NLL/celda de la respuesta bajo sft139 SIN TTT (memorizado ~0,004 | no visto ~0,05)")
print("=" * 96)
G = {}
for r in res:
    if "nll_celda" in r:
        G.setdefault(r["src"], []).append(r)
for src in ("M_agi1_train", "T2_agi2_train", "E_eval", "rearc_chico", "rearc_grande"):
    g = G.get(src, [])
    if not g: continue
    v = [r["nll_celda"] for r in g]
    print("%-15s n=%3d | nll/celda mediana %.4f p25 %.4f p75 %.4f | greedy exacto %3.0f%% | celdas med %d"
          % (src, len(g), med(v), sorted(v)[len(v) // 4], sorted(v)[3 * len(v) // 4],
             100.0 * sum(r["greedy_ok"] for r in g) / len(g), med([r["celdas"] for r in g])), flush=True)
M = med([r["nll_celda"] for r in G.get("M_agi1_train", [])])
E = med([r["nll_celda"] for r in G.get("E_eval", [])])
for src in ("rearc_chico", "rearc_grande"):
    g = G.get(src, [])
    if g and M > 0 and E > M:
        n = med([r["nll_celda"] for r in g])
        rr = math.log(max(n, 1e-9) / M) / math.log(E / M)
        alto = sum(1 for r in g if r["nll_celda"] >= E)
        print("r(%s) = %.2f  | puzzles con nll >= nivel eval: %d/%d" % (src, rr, alto, len(g)), flush=True)
print("PRERREG: r>=0,5 no memorizados (SFT viable) | 0,2-0,5 parcial (filtrar por puzzle) | <=0,2 memorizados")
print("SANIDAD: M <= ~0,01 y E ~ 0,05; si no, el banco esta mal construido.")

# comparacion PAREADA: misma tarea, misma regla, mismas demos; solo cambia la escala del test
by = {}
for r in res:
    if r["src"].startswith("rearc") and "nll_celda" in r:
        by.setdefault(r["id"].split("_")[0], {})[r["src"]] = r
pares = [(v["rearc_chico"]["nll_celda"], v["rearc_grande"]["nll_celda"]) for v in by.values() if len(v) == 2]
if pares:
    peor = sum(1 for c, g in pares if g > c)
    print("PAREADO: %d tareas | grande peor que chico en %d (%.0f%%) | mediana chico %.4f grande %.4f"
          % (len(pares), peor, 100.0 * peor / len(pares), med([c for c, _ in pares]), med([g for _, g in pares])), flush=True)
    gok = [v["rearc_grande"]["greedy_ok"] for v in by.values() if len(v) == 2]
    cok = [v["rearc_chico"]["greedy_ok"] for v in by.values() if len(v) == 2]
    print("greedy exacto pareado: chico %.0f%% | grande %.0f%%" % (100.0 * sum(cok) / len(cok), 100.0 * sum(gok) / len(gok)), flush=True)
